# CGR-MAT v1 — Leakage-Safe Multimodal Pediatric Appendicitis Severity Pipeline

**Architecture:** quality-aware multiview ultrasound encoder + leakage-safe CatBoost base expert + residual attention transformer + modality dropout.

**Evaluation:** fixed patient-level official holdout; development-only nested cross-fitting, calibration and threshold selection. The holdout is evaluated once at the end.

**Before running:** choose `Runtime → Change runtime type → T4 GPU`, then use `Runtime → Run all`. Allow Google Drive access so checkpoints and results survive runtime disconnects. No manual dataset upload or Kaggle token is required.


In [ ]:
# Install only the packages not guaranteed by the Colab base image.
!pip -q install catboost==1.2.8 openpyxl==3.1.5


In [ ]:
import os
import torch

# Use "smoke" for a fast structural test, "quick" for a shorter experiment,
# and "full" for the prespecified scientific run.
os.environ["CGR_MAT_RUN_MODE"] = "full"
os.environ["CGR_MAT_USE_DRIVE"] = "1"
os.environ["CGR_MAT_FORCE_RESTART"] = "0"
os.environ["CGR_MAT_PRETRAINED"] = "1"

if os.environ["CGR_MAT_RUN_MODE"] == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full mode requires a GPU runtime. Select Runtime → Change runtime type → T4 GPU.")
print("Run mode:", os.environ["CGR_MAT_RUN_MODE"])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import hashlib
import urllib.request

SOURCE_COMMIT = "01a080ad6cbfa2b07dee6b8d496be95ca11a791a"
LOADER_URL = (
    "https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/"
    f"{SOURCE_COMMIT}/src/cgr_mat/cgr_mat_verified_loader.py"
)
EXPECTED_LOADER_SHA256 = "b4f42bd7b5a082cf944067596c13b94b47d92eb4a925b8def5817d39f7ecd514"

print("Loading pinned CGR-MAT loader...")
print("Source commit:", SOURCE_COMMIT)
loader_bytes = urllib.request.urlopen(LOADER_URL, timeout=120).read()
actual_loader_sha256 = hashlib.sha256(loader_bytes).hexdigest()
if actual_loader_sha256 != EXPECTED_LOADER_SHA256:
    raise RuntimeError(
        "Loader integrity failure.\n"
        f"Expected: {EXPECTED_LOADER_SHA256}\nActual:   {actual_loader_sha256}"
    )
print("✓ Loader integrity verified.")
loader = loader_bytes.decode("utf-8")
exec(compile(loader, LOADER_URL, "exec"), globals(), globals())


## Saved outputs

The pipeline prints and displays the dataset audit, fixed split, live fold/epoch progress, CV summary, official holdout table, ROC, PR, calibration, confusion matrix and feature importance. It saves `.pt`, `.pkl`, `.cbm`, `.csv`, `.json`, `.png`, checksums and a ZIP bundle under `MyDrive/MAT-Appendix/cgr_mat_runs/`.
